In [4]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

In [6]:
# RAG Architecture Overview
print("""
RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge
""")


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge



In [7]:
## create sample documents
sample_docs = [
    """
    
    1. Rigveda (c. 1500 BCE – 1200 BCE)

    The Rigveda is the oldest known Vedic Sanskrit text and the foundational scripture of Hinduism.
    It is a monumental collection of 1,028 hymns (suktas) containing 10,600 verses organized into 10 books called Mandalas.
    The text was orally composed by various priestly families in the northwestern region of the Indian subcontinent.
    Most of its early hymns are dedicated to cosmic deities representing nature, such as Agni, Indra, and Soma.
    It contains profound philosophical inquiries into cosmology, the origin of the universe, and absolute virtue.
    The famous Gayatri Mantra, a highly revered sacred verse, is structurally recorded within its third Mandala.
    Linguistic and philological evidence establishes its main compilation period between 1500 BCE and 1200 BCE.
    Like all Vedas, it was strictly preserved using intricate oral mnemonic techniques before being written down.
    Its structural layers are divided into the Samhitas, Brahmanas, Aranyakas, and the highly philosophical Upanishads.
    The Rigveda provides deep historical insights into early Indo-Aryan social structures, tribal identities, and geography.
    Later additions within Mandalas 1 and 10 introduce complex metaphysical themes and reference early healing concepts.
    Thirty surviving 19th-century manuscripts of this text are preserved under UNESCO's Memory of the World Register.
    """,
    
    """
    
    2. Samaveda (c. 1200 BCE – 800 BCE)

    The Samaveda is universally recognized as the Veda of melodies and chants, focusing on the musicality of devotion.
    It was structurally compiled directly following the primary Rigvedic era, dating roughly between 1200 BCE and 800 BCE.
    The core Samhita contains 1,549 liturgical verses, which are designed to be sung rather than just recited.
    Remarkably, except for 75 standalone verses, all of its contents are musically adapted directly from the Rigveda.
    It functions essentially as a musical score sheet intended for Udgatar priests during public sacrifices and worship.
    The text is systematically divided into two main components: the Gana (melodies) and the Archika (verse books).
    It features three major surviving regional recensions or traditions: the Kauthuma, Ranayaniya, and Jaiminiya branches.
    Historians and artists consider the melodies of the Samaveda to be the root foundation of Indian classical music.
    Embedded within its vast framework are the highly influential Chandogya Upanishad and the profound Kena Upanishad.
    The text elevates ritual sacrifices by transforming spoken prayers into complex, spiritually moving rhythmic incantations.
    It is an integral part of the traditional Trayi-Vidya, which forms the initial threefold canonical Vedic knowledge.
    The preservation of its unique musical accents and pitches represents one of humankind's oldest continuous oral traditions.
    """,
    
    """
    
    3. Yajurveda (c. 1100 BCE – 800 BCE)

    The Yajurveda translates broadly to the "Knowledge of Worship" and functions as a detailed ritual manual.
    Its composition chronologically overlaps with the Samaveda, dating to the Iron Age period between 1100 BCE and 800 BCE.
    It serves as a procedural guidebook for Adhvaryu priests executing actions before the sacred yajna (fire ritual).
    The primary text features 1,875 verses written in a combination of prose formulas and borrowed Rigvedic chants.
    It is distinctively split into two major iterations: the Krishna (Black) and the Shukla (White) Yajurveda.
    The Krishna Yajurveda features an un-arranged, blended collection where interpretive prose and verses run together.
    The Shukla Yajurveda is characterized by clear, systematically arranged verses separate from its accompanying commentaries.
    A major literary benchmark of its middle layer is the Satapatha Brahmana, an extensive commentary on the Shukla tradition.
    The text details the technical architecture of altars, geometrical patterns, and explicit steps for state ceremonies.
    It shifts the focus of Vedic culture from abstract nature worship toward structured, societal, and ritualistic execution.
    The philosophical sections of this Veda culminate in the Brihadaranyaka Upanishad and the concise, profound Isha Upanishad.
    It bridges the gap between outer physical sacrifice and the inner meditative realization of the late Vedic period.
    """,

    """
    
    4. Atharvaveda (c. 1000 BCE – 500 BCE)

    The Atharvaveda is the fourth scripture, often referred to historically as the "Knowledge of the Fire Priest".
    It is the youngest of the core Vedas, composed and consolidated primarily between 1000 BCE and 500 BCE.
    Unlike the other three Vedas, it deals extensively with the practicalities, struggles, and charms of daily human life.
    It contains a diverse assembly of 730 hymns, 6,000 verses, and various prose sentences divided into 20 books.
    The contents feature healing spells, protective incantations, domestic blessings, and formulas to ward off negative forces.
    It serves as an invaluable historical repository for ancient Indian medicine, laying early concepts for Ayurveda.
    It addresses societal issues, marital harmony, agricultural prosperity, statecraft, and the duties of a ruling king.
    The text incorporates diverse local traditions, placing it slightly outside the strict boundaries of priestly fire sacrifices.
    Philosophically, it includes deeply complex metaphysical texts, most notably the Mundaka, Mandukya, and Prasna Upanishads.
    It is also known traditionally as the Brahma Veda because it emphasizes spiritual wisdom and the science of yoga.
    Due to its unique vernacular focus, it took longer to be universally integrated into the traditional Vedic canon.
    The written form of this text, like the others, was recorded on perishable palm leaves and birch bark centuries later.
    """
]

sample_docs


["\n\n    1. Rigveda (c. 1500 BCE – 1200 BCE)\n\n    The Rigveda is the oldest known Vedic Sanskrit text and the foundational scripture of Hinduism.\n    It is a monumental collection of 1,028 hymns (suktas) containing 10,600 verses organized into 10 books called Mandalas.\n    The text was orally composed by various priestly families in the northwestern region of the Indian subcontinent.\n    Most of its early hymns are dedicated to cosmic deities representing nature, such as Agni, Indra, and Soma.\n    It contains profound philosophical inquiries into cosmology, the origin of the universe, and absolute virtue.\n    The famous Gayatri Mantra, a highly revered sacred verse, is structurally recorded within its third Mandala.\n    Linguistic and philological evidence establishes its main compilation period between 1500 BCE and 1200 BCE.\n    Like all Vedas, it was strictly preserved using intricate oral mnemonic techniques before being written down.\n    Its structural layers are divided

In [9]:
os.makedirs("vedassample/",exist_ok=True)
for i,doc in enumerate(sample_docs):
    with open(f"vedassample/doc_{i}.txt","w") as f:
        f.write(doc)

In [17]:
# Document Loading 
from langchain_community.document_loaders import DirectoryLoader,TextLoader

loader = DirectoryLoader(
    "vedassample",
    glob="*.txt",
    loader_cls = TextLoader,
    loader_kwargs = {'encoding':'utf-8'}
)
documents = loader.load()

print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")

Loaded 4 documents

First document preview:
1. Rigveda (c. 1500 BCE - 1200 BCE)

    The Rigveda is the oldest known Vedic Sanskrit text and the foundational scripture of Hinduism.
    It is a monumental collection of 1,028 hymns (suktas) conta...


In [18]:
documents

[Document(metadata={'source': 'vedassample\\doc_0.txt'}, page_content="1. Rigveda (c. 1500 BCE - 1200 BCE)\n\n    The Rigveda is the oldest known Vedic Sanskrit text and the foundational scripture of Hinduism.\n    It is a monumental collection of 1,028 hymns (suktas) containing 10,600 verses organized into 10 books called Mandalas.\n    The text was orally composed by various priestly families in the northwestern region of the Indian subcontinent.\n    Most of its early hymns are dedicated to cosmic deities representing nature, such as Agni, Indra, and Soma.\n    It contains profound philosophical inquiries into cosmology, the origin of the universe, and absolute virtue.\n    The famous Gayatri Mantra, a highly revered sacred verse, is structurally recorded within its third Mandala.\n    Linguistic and philological evidence establishes its main compilation period between 1500 BCE and 1200 BCE.\n    Like all Vedas, it was strictly preserved using intricate oral mnemonic techniques befo

In [19]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # Maximum size of each chunk
    chunk_overlap=50,  # Overlap between chunks to maintain context
    length_function=len,
    separators=[" "]  # Hierarchy of separators
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 16 chunks from 4 documents

Chunk example:
Content: 1. Rigveda (c. 1500 BCE - 1200 BCE)

    The Rigveda is the oldest known Vedic Sanskrit text and the foundational scripture of Hinduism.
    It is a m...
Metadata: {'source': 'vedassample\\doc_0.txt'}


In [ ]:
### Huggingface
## Initialize a simple Embedding model(no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3569.59it/s]


HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [23]:
sample_text = "hi this is just a test"
vector=embeddings.embed_query(sample_text)
vector

[0.0034020908642560244,
 -0.020270487293601036,
 -0.009133699350059032,
 0.003827011911198497,
 0.021786853671073914,
 -0.029920775443315506,
 -0.03326210752129555,
 0.007370545994490385,
 0.010637473315000534,
 0.07416046410799026,
 0.011231942102313042,
 -0.06507696956396103,
 -0.004633581265807152,
 0.008257636800408363,
 -0.014651714824140072,
 -0.038587965071201324,
 0.03597093001008034,
 -0.11565350741147995,
 -0.1301041841506958,
 0.005660746246576309,
 0.04350139573216438,
 0.026250936090946198,
 -0.12432363629341125,
 0.005113463848829269,
 -0.06090350076556206,
 -0.01937379315495491,
 -0.04664502292871475,
 0.03247040882706642,
 0.10033156722784042,
 -0.008157476782798767,
 0.021527569741010666,
 0.07555154711008072,
 0.07213456183671951,
 -0.00161427550483495,
 -0.005706912837922573,
 -0.01052238792181015,
 0.0464036799967289,
 -0.010426636785268784,
 0.0040578050538897514,
 0.0042432998307049274,
 -0.005254208110272884,
 -0.07662113010883331,
 0.0666964203119278,
 0.0387931

In [24]:
chunks

[Document(metadata={'source': 'vedassample\\doc_0.txt'}, page_content='1. Rigveda (c. 1500 BCE - 1200 BCE)\n\n    The Rigveda is the oldest known Vedic Sanskrit text and the foundational scripture of Hinduism.\n    It is a monumental collection of 1,028 hymns (suktas) containing 10,600 verses organized into 10 books called Mandalas.\n    The text was orally composed by various priestly families in the northwestern region of the Indian subcontinent.\n    Most of its early hymns are dedicated to cosmic deities representing nature, such as Agni, Indra, and Soma.\n    It'),
 Document(metadata={'source': 'vedassample\\doc_0.txt'}, page_content='nature, such as Agni, Indra, and Soma.\n    It contains profound philosophical inquiries into cosmology, the origin of the universe, and absolute virtue.\n    The famous Gayatri Mantra, a highly revered sacred verse, is structurally recorded within its third Mandala.\n    Linguistic and philological evidence establishes its main compilation period be

In [25]:
## Create a Chromdb vector store
persist_directory="./chroma_db"

## Initialize Chromadb with Open AI embeddings
vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=HuggingFaceEmbeddings(),
    persist_directory=persist_directory,
    collection_name="rag_collection"

)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

d:\Rag Vyuham\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
d:\Rag Vyuham\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cac

Vector store created with 16 vectors
Persisted to: ./chroma_db


In [28]:
query="what is the oldest known Vedic Sanskrit text"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'vedassample\\doc_0.txt'}, page_content='1. Rigveda (c. 1500 BCE - 1200 BCE)\n\n    The Rigveda is the oldest known Vedic Sanskrit text and the foundational scripture of Hinduism.\n    It is a monumental collection of 1,028 hymns (suktas) containing 10,600 verses organized into 10 books called Mandalas.\n    The text was orally composed by various priestly families in the northwestern region of the Indian subcontinent.\n    Most of its early hymns are dedicated to cosmic deities representing nature, such as Agni, Indra, and Soma.\n    It'),
 Document(metadata={'source': 'vedassample\\doc_0.txt'}, page_content="techniques before being written down.\n    Its structural layers are divided into the Samhitas, Brahmanas, Aranyakas, and the highly philosophical Upanishads.\n    The Rigveda provides deep historical insights into early Indo-Aryan social structures, tribal identities, and geography.\n    Later additions within Mandalas 1 and 10 introduce complex met

In [29]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: what is the oldest known Vedic Sanskrit text

Top 3 similar chunks:

--- Chunk 1 ---
1. Rigveda (c. 1500 BCE - 1200 BCE)

    The Rigveda is the oldest known Vedic Sanskrit text and the foundational scripture of Hinduism.
    It is a monumental collection of 1,028 hymns (suktas) conta...
Source: vedassample\doc_0.txt

--- Chunk 2 ---
techniques before being written down.
    Its structural layers are divided into the Samhitas, Brahmanas, Aranyakas, and the highly philosophical Upanishads.
    The Rigveda provides deep historical i...
Source: vedassample\doc_0.txt

--- Chunk 3 ---
3. Yajurveda (c. 1100 BCE - 800 BCE)

    The Yajurveda translates broadly to the "Knowledge of Worship" and functions as a detailed ritual manual.
    Its composition chronologically overlaps with th...
Source: vedassample\doc_2.txt


In [30]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[(Document(metadata={'source': 'vedassample\\doc_0.txt'}, page_content='1. Rigveda (c. 1500 BCE - 1200 BCE)\n\n    The Rigveda is the oldest known Vedic Sanskrit text and the foundational scripture of Hinduism.\n    It is a monumental collection of 1,028 hymns (suktas) containing 10,600 verses organized into 10 books called Mandalas.\n    The text was orally composed by various priestly families in the northwestern region of the Indian subcontinent.\n    Most of its early hymns are dedicated to cosmic deities representing nature, such as Agni, Indra, and Soma.\n    It'),
  0.5681041479110718),
 (Document(metadata={'source': 'vedassample\\doc_0.txt'}, page_content="techniques before being written down.\n    Its structural layers are divided into the Samhitas, Brahmanas, Aranyakas, and the highly philosophical Upanishads.\n    The Rigveda provides deep historical insights into early Indo-Aryan social structures, tribal identities, and geography.\n    Later additions within Mandalas 1 and